# 9.2 Training Dynamics — Initialization, Normalization, Dropout, BatchNorm, Optimizers

**Phase 9 — Deep Learning Foundations | Section 2 of 4**

Section 9.1 proved backpropagation works. This section covers everything that determines
whether it works WELL — the difference between a network that trains smoothly and one that
stalls, explodes, or memorizes noise:

1. **Weight initialization** — why all-zero weights break learning entirely, and what
   Xavier/He initialization fix
2. **Optimization** — SGD vs. SGD+momentum vs. Adam, same architecture, same data
3. **BatchNorm** — why normalizing activations mid-network stabilizes deeper training
4. **Dropout** — the direct fix for overfitting on a small training set

### Data
[MNIST](https://raw.githubusercontent.com/fgnt/mnist/master/train-images-idx3-ubyte.gz) —
handwritten digit images, loaded directly from GitHub as the original IDX binary format (the
same format the dataset has shipped in since 1998), decoded by hand below rather than via a
bundled dataset loader. A 10,000-image subset keeps these ablations fast enough to compare
several variants side by side.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import gzip
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(42)
np.random.seed(42)

def load_idx_gz(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as resp:
        raw = gzip.decompress(resp.read())
    magic = int.from_bytes(raw[0:4], "big")
    n_dims = magic & 0xFF
    dims = [int.from_bytes(raw[4 + 4*i : 8 + 4*i], "big") for i in range(n_dims)]
    data = np.frombuffer(raw[4 + 4*n_dims:], dtype=np.uint8)
    return data.reshape(dims)

images = load_idx_gz("https://raw.githubusercontent.com/fgnt/mnist/master/train-images-idx3-ubyte.gz")
labels = load_idx_gz("https://raw.githubusercontent.com/fgnt/mnist/master/train-labels-idx1-ubyte.gz")
print(images.shape, labels.shape)

In [ ]:
N_SUBSET = 10_000
rng = np.random.default_rng(42)
subset_idx = rng.choice(len(images), N_SUBSET, replace=False)
X = images[subset_idx].reshape(N_SUBSET, -1).astype(np.float32) / 255.0
y = labels[subset_idx].astype(np.int64)

split = int(N_SUBSET * 0.8)
X_train, X_val = torch.tensor(X[:split]), torch.tensor(X[split:])
y_train, y_val = torch.tensor(y[:split]), torch.tensor(y[split:])
print(f"train: {X_train.shape}, val: {X_val.shape}")

fig, axes = plt.subplots(1, 6, figsize=(11, 2))
for i, ax in enumerate(axes):
    ax.imshow(X_train[i].reshape(28, 28), cmap="gray"); ax.set_title(int(y_train[i])); ax.axis("off")
plt.tight_layout(); plt.show()

## 1. Weight initialization: why all-zero weights break learning

In [ ]:
def make_mlp(init_fn=None):
    model = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 10))
    if init_fn is not None:
        for layer in model:
            if isinstance(layer, nn.Linear):
                init_fn(layer.weight)
                nn.init.zeros_(layer.bias)
    return model

def train_quick(model, epochs=15, lr=0.1):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    history = []
    for _ in range(epochs):
        optimizer.zero_grad()
        out = model(X_train)
        loss = loss_fn(out, y_train)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    return history

zero_model = make_mlp(init_fn=nn.init.zeros_)
zero_history = train_quick(zero_model)

xavier_model = make_mlp(init_fn=nn.init.xavier_uniform_)
xavier_history = train_quick(xavier_model)

he_model = make_mlp(init_fn=lambda w: nn.init.kaiming_uniform_(w, nonlinearity="relu"))
he_history = train_quick(he_model)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(zero_history, label="all-zero init")
ax.plot(xavier_history, label="Xavier/Glorot init")
ax.plot(he_history, label="He init (ReLU-aware)")
ax.set_xlabel("step"); ax.set_ylabel("train loss"); ax.legend()
ax.set_title("Same architecture, same data -- only the INITIALIZATION differs")
plt.tight_layout(); plt.show()

> ⚠️ **All-zero init should barely move.** Every neuron in a layer starts identical and
> receives the identical gradient update forever (the "symmetry problem") — the network
> effectively has ONE useful neuron per layer no matter how wide it is. Xavier and He
> initialization break that symmetry with carefully SCALED randomness (matched to the
> activation function and layer width) so gradients neither vanish nor explode in the first
> few steps.

## 2. Optimizers: SGD vs. SGD+momentum vs. Adam

In [ ]:
def train_full(model, optimizer, epochs=40):
    loss_fn = nn.CrossEntropyLoss()
    history = []
    for _ in range(epochs):
        optimizer.zero_grad()
        out = model(X_train)
        loss = loss_fn(out, y_train)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    return history

torch.manual_seed(1); sgd_model = make_mlp(init_fn=lambda w: nn.init.kaiming_uniform_(w, nonlinearity="relu"))
sgd_history = train_full(sgd_model, torch.optim.SGD(sgd_model.parameters(), lr=0.1))

torch.manual_seed(1); momentum_model = make_mlp(init_fn=lambda w: nn.init.kaiming_uniform_(w, nonlinearity="relu"))
momentum_history = train_full(momentum_model, torch.optim.SGD(momentum_model.parameters(), lr=0.1, momentum=0.9))

torch.manual_seed(1); adam_model = make_mlp(init_fn=lambda w: nn.init.kaiming_uniform_(w, nonlinearity="relu"))
adam_history = train_full(adam_model, torch.optim.Adam(adam_model.parameters(), lr=0.01))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sgd_history, label="plain SGD")
ax.plot(momentum_history, label="SGD + momentum")
ax.plot(adam_history, label="Adam")
ax.set_xlabel("step"); ax.set_ylabel("train loss"); ax.legend()
ax.set_title("Same init, same architecture -- only the OPTIMIZER differs")
plt.tight_layout(); plt.show()

> 💡 **Momentum accumulates a running average of past gradients**, smoothing out noisy
> steps and accelerating through consistent directions — think of a ball rolling downhill
> gaining speed. **Adam** adds a PER-PARAMETER adaptive learning rate on top of momentum,
> which is usually why it converges fastest with the least tuning — the reason it's the
> default choice in most modern training loops, including every PyTorch example later in
> this curriculum.

## 3. BatchNorm: stabilizing a deeper network

In [ ]:
def make_deep_mlp(use_batchnorm):
    layers = []
    sizes = [784, 256, 256, 256, 128, 10]
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i+1]))
        if i < len(sizes) - 2:
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(sizes[i+1]))
            layers.append(nn.ReLU())
    return nn.Sequential(*layers)

torch.manual_seed(2); plain_deep = make_deep_mlp(use_batchnorm=False)
plain_deep_history = train_full(plain_deep, torch.optim.SGD(plain_deep.parameters(), lr=0.5), epochs=60)

torch.manual_seed(2); bn_deep = make_deep_mlp(use_batchnorm=True)
bn_deep_history = train_full(bn_deep, torch.optim.SGD(bn_deep.parameters(), lr=0.5), epochs=60)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(plain_deep_history, label="5-layer MLP, no BatchNorm")
ax.plot(bn_deep_history, label="5-layer MLP, WITH BatchNorm")
ax.set_xlabel("step"); ax.set_ylabel("train loss"); ax.legend()
ax.set_title("A deeper network: BatchNorm's stabilizing effect on the SAME aggressive learning rate")
plt.tight_layout(); plt.show()

> 💡 BatchNorm re-centers and re-scales each layer's activations to roughly zero mean, unit
> variance, EVERY forward pass — which keeps every layer training against a stable input
> distribution instead of chasing a constantly shifting one as earlier layers' weights
> change underneath it (sometimes called "internal covariate shift"). That stability is
> exactly what lets you use a more aggressive learning rate on a deeper network without it
> diverging.

## 4. Dropout: the direct fix for overfitting on limited data

In [ ]:
N_TINY = 300  # deliberately tiny training set to induce visible overfitting
X_tiny, y_tiny = X_train[:N_TINY], y_train[:N_TINY]

def train_with_val(model, optimizer, epochs=150):
    loss_fn = nn.CrossEntropyLoss()
    train_acc_history, val_acc_history = [], []
    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        out = model(X_tiny)
        loss = loss_fn(out, y_tiny)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            train_acc = (model(X_tiny).argmax(1) == y_tiny).float().mean().item()
            val_acc = (model(X_val).argmax(1) == y_val).float().mean().item()
        train_acc_history.append(train_acc)
        val_acc_history.append(val_acc)
    return train_acc_history, val_acc_history

torch.manual_seed(3)
no_dropout_model = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 10))
no_dropout_train, no_dropout_val = train_with_val(no_dropout_model, torch.optim.Adam(no_dropout_model.parameters(), lr=1e-3))

torch.manual_seed(3)
dropout_model = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Dropout(0.5),
                                nn.Linear(256, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, 10))
dropout_train, dropout_val = train_with_val(dropout_model, torch.optim.Adam(dropout_model.parameters(), lr=1e-3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(no_dropout_train, label="train acc"); axes[0].plot(no_dropout_val, label="val acc")
axes[0].set_title(f"No dropout -- final gap: {no_dropout_train[-1]-no_dropout_val[-1]:.1%}")
axes[0].legend(); axes[0].set_ylim(0, 1.05)

axes[1].plot(dropout_train, label="train acc"); axes[1].plot(dropout_val, label="val acc")
axes[1].set_title(f"Dropout(0.5) -- final gap: {dropout_train[-1]-dropout_val[-1]:.1%}")
axes[1].legend(); axes[1].set_ylim(0, 1.05)
plt.tight_layout(); plt.show()

> ⚠️ **The train/validation GAP is the story, not the raw accuracy.** With only 300 training
> images, the no-dropout model should approach ~100% train accuracy while validation
> accuracy plateaus far lower — textbook overfitting (Phase 4's bias-variance tradeoff, now
> visible inside a single network). Dropout randomly zeroes a fraction of neurons EVERY
> forward pass during training, forcing the network to not rely on any single neuron too
> heavily — a direct, mechanical form of regularization.

## 🎯 Exercises

1. Try initializing weights with `nn.init.normal_(w, std=1.0)` (large random values, not
   scaled to the layer size) — does training diverge (loss increasing or NaN) the way it
   would for all-zero init, just for a different reason?
2. Add a learning rate scheduler (`torch.optim.lr_scheduler.StepLR`) to the Adam optimizer
   from Section 2 — does decaying the learning rate over training improve final loss?
3. Try `nn.BatchNorm1d` with a SMALL batch (e.g., train on 8 examples at a time via
   `X_tiny[:8]`) — BatchNorm's statistics get noisy with tiny batches. Does training become
   less stable?
4. Try `Dropout(0.2)` and `Dropout(0.8)` instead of `0.5` on the tiny-data experiment — is
   there a dropout rate that's clearly too aggressive (train accuracy also suffers)?

### Solutions

```python
# 1.
torch.manual_seed(5)
large_init_model = make_mlp(init_fn=lambda w: nn.init.normal_(w, std=1.0))
large_init_history = train_quick(large_init_model)
print(large_init_history)

# 2.
scheduled_model = make_mlp(init_fn=lambda w: nn.init.kaiming_uniform_(w, nonlinearity="relu"))
opt = torch.optim.Adam(scheduled_model.parameters(), lr=0.01)
scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5)

# 3.
X_micro, y_micro = X_train[:8], y_train[:8]
micro_bn_model = nn.Sequential(nn.Linear(784, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Linear(64, 10))

# 4.
for p in [0.2, 0.8]:
    torch.manual_seed(3)
    m = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Dropout(p),
                        nn.Linear(256, 256), nn.ReLU(), nn.Dropout(p), nn.Linear(256, 10))
    tr, va = train_with_val(m, torch.optim.Adam(m.parameters(), lr=1e-3))
    print(p, tr[-1], va[-1])
```